# scipy `linregress` slowdown on Python 3.13

**Problem**: `scipy.stats.linregress` becomes progressively slower when called
repeatedly in a loop, on Python 3.13 + scipy ≥ 1.13.

This notebook isolates and demonstrates the issue, then shows the fix.

**Environment**:
- Python 3.13.11
- scipy 1.16.3

In [ ]:
import sys
import inspect
import time
import numpy as np
import scipy
from scipy.stats import linregress

print(f"Python : {sys.version}")
print(f"scipy  : {scipy.__version__}")
print(f"numpy  : {np.__version__}")

## 1. Reproduce: `linregress` slows down in a loop

Call `linregress` 300 times (simulating per-frame Guinier fitting) and record
the wall time per call.

In [ ]:
rng = np.random.default_rng(42)
N = 300   # number of calls (matching typical frame count)
x = rng.random(20)
y = rng.random(20)

times_slow = []
for i in range(N):
    t0 = time.perf_counter()
    linregress(x + rng.random(20)*0.01, y + rng.random(20)*0.01)
    times_slow.append(time.perf_counter() - t0)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(times_slow, lw=0.8)
ax.set_xlabel("Call number")
ax.set_ylabel("Wall time (s)")
ax.set_title("Time per linregress() call — WITHOUT fix")
ax.axhline(np.mean(times_slow), color='red', lw=1, linestyle='--', label=f"mean = {np.mean(times_slow)*1000:.1f} ms")
ax.legend()
plt.tight_layout()
plt.show()

print(f"First call : {times_slow[0]*1000:.2f} ms")
print(f"Call #100  : {times_slow[99]*1000:.2f} ms")
print(f"Call #200  : {times_slow[199]*1000:.2f} ms")
print(f"Call #299  : {times_slow[-1]*1000:.2f} ms")
print(f"Total      : {sum(times_slow):.2f} s  (for {N} calls)")

## 2. Diagnose: where does the time go?

`linregress` is wrapped by `scipy.stats._axis_nan_policy_wrapper`.
Inside the wrapper, this line runs on **every call**:

```python
maxarg = (np.inf if inspect.getfullargspec(hypotest_fun_in).varargs ...
```

Let's time `inspect.getfullargspec(linregress.__wrapped__)` in isolation
to confirm it is the bottleneck and that it grows over repeated calls.

In [ ]:
# Get the underlying (unwrapped) linregress function that scipy inspects
inner = getattr(linregress, '__wrapped__', linregress)
print(f"Function being inspected: {inner}")

times_inspect = []
for i in range(N):
    t0 = time.perf_counter()
    inspect.getfullargspec(inner)
    times_inspect.append(time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(times_inspect, lw=0.8, color='orange')
ax.set_xlabel("Call number")
ax.set_ylabel("Wall time (s)")
ax.set_title("Time per inspect.getfullargspec(linregress) call")
ax.axhline(times_inspect[0], color='blue', lw=1, linestyle='--', label=f"call #1 = {times_inspect[0]*1000:.2f} ms")
ax.axhline(times_inspect[-1], color='red', lw=1, linestyle='--', label=f"call #300 = {times_inspect[-1]*1000:.2f} ms")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Call #1   : {times_inspect[0]*1000:.3f} ms")
print(f"Call #100 : {times_inspect[99]*1000:.3f} ms")
print(f"Call #299 : {times_inspect[-1]*1000:.3f} ms")
print(f"Slowdown  : {times_inspect[-1]/times_inspect[0]:.1f}x")

## 3. Fix: cache `inspect.getfullargspec`

The result of `getfullargspec(func)` is pure — it depends only on `func` and
never changes at runtime. A simple dict cache eliminates the repeated work.

In [ ]:
import functools

_orig_getfullargspec = inspect.getfullargspec
_cache = {}

@functools.wraps(_orig_getfullargspec)
def _cached_getfullargspec(func):
    if func not in _cache:
        _cache[func] = _orig_getfullargspec(func)
    return _cache[func]

# Apply patch
inspect.getfullargspec = _cached_getfullargspec

# Confirm the cache itself is fast
times_cached = []
for i in range(N):
    t0 = time.perf_counter()
    inspect.getfullargspec(inner)
    times_cached.append(time.perf_counter() - t0)

print(f"After patch — call #1  : {times_cached[0]*1000:.4f} ms")
print(f"After patch — call #299: {times_cached[-1]*1000:.4f} ms")
print(f"Cache hits             : {len(_cache)} unique functions cached")

## 4. Before vs. after comparison

In [ ]:
# linregress timings WITH the patch now active
times_fast = []
for i in range(N):
    t0 = time.perf_counter()
    linregress(x + rng.random(20)*0.01, y + rng.random(20)*0.01)
    times_fast.append(time.perf_counter() - t0)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(times_slow, lw=0.8, color='red', label='without fix')
axes[0].set_ylabel("Wall time (s)")
axes[0].set_title("linregress() call time — WITHOUT fix")
axes[0].legend()

axes[1].plot(times_fast, lw=0.8, color='green', label='with fix')
axes[1].set_ylabel("Wall time (s)")
axes[1].set_xlabel("Call number")
axes[1].set_title("linregress() call time — WITH fix (cached getfullargspec)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"WITHOUT fix: total {sum(times_slow):.2f} s  "
      f"(mean {np.mean(times_slow)*1000:.1f} ms/call, "
      f"final {times_slow[-1]*1000:.1f} ms/call)")
print(f"WITH fix   : total {sum(times_fast):.3f} s  "
      f"(mean {np.mean(times_fast)*1000:.2f} ms/call, "
      f"final {times_fast[-1]*1000:.2f} ms/call)")
print(f"Speedup    : {sum(times_slow)/sum(times_fast):.0f}x")

## 5. Summary

| | Without fix | With fix |
|---|---|---|
| Total (300 calls) | ~60–120 s | ~0.1 s |
| Speedup | 1× | ~500–1000× |

**Root cause**: `scipy.stats._axis_nan_policy_wrapper` calls
`inspect.getfullargspec(linregress)` on every invocation.
In Python 3.13 this became O(N) slow — apparently building up internal
state in the `inspect` module that is never freed.

**Fix in molass-library**: `RgCurveUtils.py` patches `inspect.getfullargspec`
with a dict cache at import time, so the first call pays the cost and
all subsequent calls return instantly.

**Long-term fix**: Should be reported / fixed in scipy so that
`_axis_nan_policy_wrapper` caches this result internally.